# Intro

- In this homework we will take the FAQ agent from Module 1, instrument it with [Pydantic Logfire](https://logfire.dev) for observability, then pull the trace data back out with dlt and analyze it.
- In Module 1 we wrote the agent loop by hand and then we saw toyaikit - an agentic framework.
- For this homework we rewrote into [Pydantic AI](https://ai.pydantic.dev/), so it's easier to integrate it with Logfire.
    - Pydantic AI and Logfire work really well together, that's why we use them here.
- In Module 5 we learned about monitoring and observability, and implement our own monitoring solution.
    - Logfire is an alternative for that.

# Add dependencies

```bash
uv add --active openai minsearch requests python-dotenv pydantic-ai logfire "dlt[duckdb]"
```

# Set up the source code

- [`ingest.py`](./hw_src/ingest.py) — downloads the course FAQ and builds the search index
- [`agent.py`](./hw_src/agent.py) — the FAQ agent built with Pydantic AI (system prompt + search tool)
- [`main.py`](./hw_src/main.py) — entry point that wires everything together

# Verify that the agent runs

```bash
uv run --active python hw_src/main.py
```

# Question 1. Instrument the agent with Logfire

- Sign up for a free [Logfire](https://logfire.dev) account, create a project, and generate a write token. Put it in `.env` as `LOGFIRE_TOKEN`.
- Instrument the agent:
    ```python
    logfire.configure()
    logfire.instrument_pydantic_ai()
    ```
- Run the agent a few times with different questions and open your project on Logfire to see the traces.

- For the following query how many spans does a single agent run produce?
    > How do I run Ollama locally?
    ```bash
    uv run --active python hw_src/main.py "How do I run Ollama locally?"
    ```
- Each span is either the agent run itself, an LLM call, or a tool call.
- The number can vary between runs because the model decides how many times to search.

* 1
* 5
* 15
* 30

# Question 2. Load traces into DuckDB with dlt

- Generate a read token for your Logfire project and set it as `LOGFIRE_READ_TOKEN` in `.env`.
- Initialize a dlt-hub project like in the workshop. Then ask your coding agent to pull the data from Pydantic Logfire and save it into DuckDB.
    - The dltHub AI workbench has a ready-made context for Logfire. Point your agent to it: https://dlthub.com/context/source/logfire
- The logfire traces contain deeply nested JSON (span attributes with LLM messages, tool calls, token usage, etc.).
    - dlt automatically normalizes this into a set of tables - one for the main records, plus child tables for each nested level.

- We run the [pipeline](./logfire_pipeline.py) using:
    ```bash
    uv run --active python logfire_pipeline.py
    ```

- How many tables did dlt create?
- Check with:
    ```sql
    SELECT COUNT(*) FROM information_schema.tables 
    WHERE table_schema = 'agent_traces';
    ```

In [ ]:
import duckdb
con = duckdb.connect('.dlt/data/dev/logfire_pipeline.duckdb', read_only=True)
query = "SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'agent_traces'"
print(con.execute(query).fetchone())

* 1
* 3
* 24
* 100

# Question 3. Query traces with an agent

- Using a coding agent (you can also write the code by hand) find the input token usage for the agent run from Q1.
- The token counts are stored in the span attributes as `gen_ai.usage.input_tokens`.
- Sum them across all LLM calls within the trace. The number depends on how many searches the agent made, so report the range it falls into.

In [ ]:
query = """
SELECT DISTINCT r.trace_id
FROM agent_traces.records__attributes__gen_ai_input_messages__parts p
JOIN agent_traces.records__attributes__gen_ai_input_messages m ON p._dlt_parent_id = m._dlt_id
JOIN agent_traces.records r ON m._dlt_parent_id = r._dlt_id
WHERE p.type = 'text' AND trim(p.content) = 'How do I run Ollama locally?'
"""
print(con.execute(query).fetchall())

In [ ]:
q = '''
WITH q1_traces AS (
    SELECT DISTINCT r.trace_id
    FROM agent_traces.records__attributes__gen_ai_input_messages__parts p
    JOIN agent_traces.records__attributes__gen_ai_input_messages m ON p._dlt_parent_id = m._dlt_id
    JOIN agent_traces.records r ON m._dlt_parent_id = r._dlt_id
    WHERE p.type = 'text' AND trim(p.content) = 'How do I run Ollama locally?'
),
per_trace_sum AS (
    SELECT rec.trace_id, SUM(rec.attributes__gen_ai_usage_input_tokens) AS input_tokens
    FROM agent_traces.records rec
    JOIN q1_traces t ON rec.trace_id = t.trace_id
    WHERE rec.span_name = 'chat gpt-5.4-mini'
    GROUP BY rec.trace_id
)
SELECT AVG(input_tokens) AS avg_input_tokens, COUNT(*) AS num_runs
FROM per_trace_sum
'''
print(con.execute(q).fetchdf())

* 100 - 500
* 1500 - 5000
* 10000 - 20000
* 50000 - 100000

# Question 1 - Revisited

- Now that we have access to duckDB table, we can also query for the average number of spans for Q1 questions we asked

In [ ]:
q = '''
WITH q1_traces AS (
    SELECT DISTINCT r.trace_id
    FROM agent_traces.records__attributes__gen_ai_input_messages__parts p
    JOIN agent_traces.records__attributes__gen_ai_input_messages m ON p._dlt_parent_id = m._dlt_id
    JOIN agent_traces.records r ON m._dlt_parent_id = r._dlt_id
    WHERE p.type = 'text' AND trim(p.content) = 'How do I run Ollama locally?'
),
per_trace_span_count AS (
    SELECT rec.trace_id, COUNT(*) AS num_spans
    FROM agent_traces.records rec
    JOIN q1_traces t ON rec.trace_id = t.trace_id
    GROUP BY rec.trace_id
)
SELECT AVG(num_spans) AS avg_spans, COUNT(*) AS num_runs
FROM per_trace_span_count
'''
print(con.execute(q).fetchdf())